# Lab Data Processing for HiBEHRT

This notebook demonstrates how to process MIMIC-IV lab data for the HiBEHRT hierarchical transformer model.

**Note**: This is separate from text processing (discharge notes, radiology reports) which uses BioClinical Modern BERT.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.append('../src')

# Import lab processing functions
from data.lab_features_hibehrt import (
    process_lab_events_for_hibehrt,
    create_lab_sequences_by_admission,
    save_lab_sequences_hibehrt,
    get_lab_summary_stats
)
from data.integrate_lab_features import (
    prepare_lab_features_for_hibehrt,
    validate_lab_features_for_hibehrt
)

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load Data

Load the cohort admissions and corresponding lab events.

In [ ]:
# Load cohort data (from previous notebook steps)
# This should be your filtered admissions from the cohort selection
admissions_df = pd.read_parquet('../data/interim/cohort.parquet')
print(f"Loaded {len(admissions_df)} admissions from cohort")
print(f"Columns: {list(admissions_df.columns)}")
print(f"Sample admission:")
print(admissions_df.head())

In [ ]:
# Load lab events
# Note: This is a large file, consider using nrows for initial testing
labevents_path = '../physionet.org/files/mimiciv/3.1/hosp/labevents.csv.gz'
labitems_path = '../physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz'

# Load lab items to get labels
labitems_df = pd.read_csv(labitems_path, compression='gzip')
print(f"Loaded {len(labitems_df)} lab items")
print(f"Lab items columns: {list(labitems_df.columns)}")
print(labitems_df[['itemid', 'label', 'category']].head())

In [ ]:
# For initial testing, load a sample of lab events
# Remove nrows parameter for full processing
labevents_df = pd.read_csv(labevents_path, compression='gzip', nrows=100000)
print(f"Loaded {len(labevents_df)} lab events (sample)")

# Merge with lab items to get labels
labevents_df = labevents_df.merge(
    labitems_df[['itemid', 'label', 'fluid', 'category', 'loinc_code']], 
    on='itemid', how='left'
)
print(f"Merged lab events shape: {labevents_df.shape}")
print(f"Lab events with labels: {labevents_df['label'].notna().sum()}")

## 2. Filter to Cohort Admissions

Only keep lab events for admissions in our cohort.

In [ ]:
# Filter lab events to cohort admissions
cohort_hadm_ids = set(admissions_df['hadm_id'].astype(str))
labevents_cohort = labevents_df[labevents_df['hadm_id'].astype(str).isin(cohort_hadm_ids)].copy()

print(f"Lab events for cohort: {len(labevents_cohort)} ({len(labevents_cohort)/len(labevents_df)*100:.1f}% of total)")
print(f"Unique admissions with labs: {labevents_cohort['hadm_id'].nunique()}")
print(f"Cohort admissions: {len(cohort_hadm_ids)}")
print(f"Lab coverage: {labevents_cohort['hadm_id'].nunique()/len(cohort_hadm_ids)*100:.1f}%")

## 3. Process Lab Events for HiBEHRT

Convert lab events into HiBEHRT-compatible format with tokenization and time sequencing.

In [ ]:
# Process lab events
print("Processing lab events for HiBEHRT format...")

processed_labs, code_to_id, id_to_code = process_lab_events_for_hibehrt(
    labevents_cohort, 
    admissions_df,
    max_events_per_admission=2048,
    time_window_hours=6,
    min_frequency=10
)

print(f"Processed labs shape: {processed_labs.shape}")
print(f"Vocabulary size: {len(code_to_id)}")
print(f"Top 10 lab codes:")
for i, (code, code_id) in enumerate(list(code_to_id.items())[:10]):
    print(f"  {code_id:3d}: {code}")

In [ ]:
# Check lab type distribution
lab_type_counts = processed_labs['label'].value_counts()
print(f"\nTop 15 lab types by frequency:")
print(lab_type_counts.head(15))

# Plot distribution
plt.figure(figsize=(12, 6))
lab_type_counts.head(20).plot(kind='bar')
plt.title('Top 20 Lab Types by Frequency')
plt.xlabel('Lab Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Create Lab Sequences by Admission

Group lab events into time-sequenced sequences for each admission.

In [ ]:
# Create sequences
print("Creating lab sequences by admission...")

lab_sequences = create_lab_sequences_by_admission(
    processed_labs, 
    admissions_df,
    max_events_per_admission=2048,
    time_window_hours=6
)

print(f"Created sequences for {len(lab_sequences)} admissions")

# Get statistics
stats = get_lab_summary_stats(lab_sequences)
print(f"\nLab Sequence Statistics:")
print(f"  Total admissions with labs: {stats['total_admissions']}")
print(f"  Total lab events: {stats['total_lab_events']}")
print(f"  Mean events per admission: {stats['mean_events_per_admission']:.1f}")
print(f"  Median events per admission: {stats['median_events_per_admission']:.1f}")
print(f"  Admissions with >100 lab events: {sum(1 for count in [len(seq) for seq in lab_sequences.values()] if count > 100)}")

In [ ]:
# Visualize lab event distribution
event_counts = [len(seq) for seq in lab_sequences.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Histogram of events per admission
ax1.hist(event_counts, bins=50, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Number of Lab Events per Admission')
ax1.set_ylabel('Number of Admissions')
ax1.set_title('Distribution of Lab Events per Admission')
ax1.axvline(np.mean(event_counts), color='red', linestyle='--', label=f'Mean: {np.mean(event_counts):.1f}')
ax1.axvline(np.median(event_counts), color='orange', linestyle='--', label=f'Median: {np.median(event_counts):.1f}')
ax1.legend()

# Box plot
ax2.boxplot(event_counts, vert=True)
ax2.set_ylabel('Number of Lab Events per Admission')
ax2.set_title('Box Plot: Lab Events per Admission')
ax2.set_xticklabels(['All Admissions'])

plt.tight_layout()
plt.show()

## 5. Example Sequence Inspection

Look at a sample lab sequence to understand the format.

In [ ]:
# Get a sample admission with many labs
admission_with_most_labs = max(lab_sequences.keys(), key=lambda x: len(lab_sequences[x]))
sample_sequence = lab_sequences[admission_with_most_labs]

print(f"Sample admission: {admission_with_most_labs}")
print(f"Total lab events: {len(sample_sequence)}")
print(f"\nFirst 5 lab events in sequence:")
for i, event in enumerate(sample_sequence[:5]):
    print(f"  {i+1}. {event['token_str']:40s} | Value: {event['value']:8.2f} | Time: {event['hours_from_admission']:6.1f}h | Window: {event['time_window']}")

print(f"\nLast 5 lab events in sequence:")
for i, event in enumerate(sample_sequence[-5:]):
    print(f"  {len(sample_sequence)-5+i+1}. {event['token_str']:40s} | Value: {event['value']:8.2f} | Time: {event['hours_from_admission']:6.1f}h | Window: {event['time_window']}")

## 6. Save Lab Features for HiBEHRT

Save the processed lab features in the format expected by HiBEHRT.

In [ ]:
# Create output directory
output_dir = Path('../data/processed/structured')
output_dir.mkdir(parents=True, exist_ok=True)

# Save lab features
feature_path = output_dir / 'lab_features_hibehrt.pkl'
vocab_path = output_dir / 'lab_vocabulary.txt'
stats_path = output_dir / 'lab_statistics.json'

save_lab_sequences_hibehrt(
    lab_sequences, 
    (code_to_id, id_to_code), 
    str(feature_path), 
    split_type='all'
)

# Save vocabulary for inspection
with open(vocab_path, 'w') as f:
    f.write(f"Lab Vocabulary for HiBEHRT\n")
    f.write(f"Total codes: {len(code_to_id)}\n")
    f.write("="*50 + "\n")
    for code, code_id in sorted(code_to_id.items(), key=lambda x: x[1]):
        f.write(f"{code_id:4d}: {code}\n")

# Save statistics
import json
with open(stats_path, 'w') as f:
    json.dump({
        'statistics': stats,
        'vocab_size': len(code_to_id),
        'coverage_percentage': stats['total_admissions']/len(admissions_df)*100
    }, f, indent=2)

print(f"Lab features saved:")
print(f"  Features: {feature_path}")
print(f"  Vocabulary: {vocab_path}")
print(f"  Statistics: {stats_path}")

## 7. Validate Features for HiBEHRT

Ensure the lab features are properly formatted for HiBEHRT input.

In [ ]:
# Validate lab features
validation_results = validate_lab_features_for_hibehrt(str(feature_path))

print("Validation Results:")
print(f"  Valid: {validation_results['is_valid']}")
print(f"  Errors: {len(validation_results['errors'])}")
print(f"  Warnings: {len(validation_results['warnings'])}")

if validation_results['errors']:
    print("\nErrors found:")
    for error in validation_results['errors']:
        print(f"  - {error}")

if validation_results['warnings']:
    print("\nWarnings:")
    for warning in validation_results['warnings']:
        print(f"  - {warning}")

print("\nValidation details:")
for detail in validation_results['required_keys']:
    print(f"  {detail}")

## 8. Integration with Text Processing

Show how lab features integrate with text processing (separate pipeline).

In [ ]:
# Load lab features for integration
from lab_features_hibehrt import load_lab_hibehrt_features

lab_features = load_lab_hibehrt_features(str(feature_path))
lab_sequences = lab_features['sequences']
lab_vocab = lab_features['vocabulary']

print(f"Loaded lab features:")
print(f"  Admissions: {len(lab_sequences)}")
print(f"  Vocabulary size: {lab_vocab['vocab_size']}")
print(f"  Total events: {lab_features['metadata']['total_events']}")

# Show how this integrates with admissions that have text
# (Text processing would be done separately with BioClinical BERT)
admissions_with_labs = set(lab_sequences.keys())
print(f"\nAdmissions with lab data: {len(admissions_with_labs)}")
print(f"This will be merged with text features in the fusion step")

# Example of how to check alignment with text data
# text_features = load_text_features(...)  # From text processing
# admissions_with_text = set(text_features['sequences'].keys())
# admissions_with_both = admissions_with_labs.intersection(admissions_with_text)
# print(f"Admissions with both lab and text: {len(admissions_with_both)}")

## 9. Summary and Next Steps

Summary of lab processing and integration with the overall pipeline.

In [ ]:
# Summary
print("LAB DATA PROCESSING SUMMARY")
print("="*50)
print(f"✓ Processed {len(processed_labs)} lab measurements")
print(f"✓ Created vocabulary with {len(code_to_id)} lab codes")
print(f"✓ Generated sequences for {len(lab_sequences)} admissions")
print(f"✓ Saved features to: {feature_path}")
print(f"✓ Validation passed: {validation_results['is_valid']}")

print("\nNext steps:")
print("1. Process text data (discharge notes, radiology reports) with BioClinical Modern BERT")
print("2. Create structured EHR features (procedures, diagnoses, vitals) for HiBEHRT")
print("3. Train HiBEHRT model on lab + structured features")
print("4. Train BioClinical BERT on text features")
print("5. Fuse modalities and train final model")

print(f"\nLab features ready for HiBEHRT model training!")
print(f"Feature file: {feature_path}")
print(f"Vocabulary file: {vocab_path}")
print(f"Statistics file: {stats_path}")

## Appendix: Configuration Options

Different configuration options for lab processing.

In [ ]:
# Example configurations

# Configuration 1: High-resolution (hourly windows, large vocab)
config_high_res = {
    'max_events_per_admission': 2048,
    'time_window_hours': 1,  # Hourly granularity
    'min_frequency': 5,
    'include_abnormal_flags': True
}

# Configuration 2: Balanced (6-hour windows, moderate vocab)
config_balanced = {
    'max_events_per_admission': 1024,
    'time_window_hours': 6,  # 6-hour granularity
    'min_frequency': 10,
    'include_abnormal_flags': True
}

# Configuration 3: Low-resolution (daily windows, small vocab)
config_low_res = {
    'max_events_per_admission': 512,
    'time_window_hours': 24,  # Daily granularity
    'min_frequency': 20,
    'include_abnormal_flags': False
}

print("Available configurations:")
for name, config in [('High Resolution', config_high_res), ('Balanced', config_balanced), ('Low Resolution', config_low_res)]:
    print(f"\n{name}:")
    for key, value in config.items():
        print(f"  {key}: {value}")